# SI Figure — factorised vs. non-factorised image-formation model

The production simulation factorises the wavelength dependence: a single
mean-wavelength PSF width $\sigma(\bar\lambda)$ for every sampled photon, and a separately
spectrum-averaged detector response $QE_{d,n}=\int p(\lambda)QE_n(\lambda)d\lambda$. This
notebook compares that production model (A) against two increasingly rigorous references: (B)
a non-factorised, photon-by-photon model (each photon gets its own wavelength-dependent PSF
width *and* detection probability, but the PSF is still a Gaussian), and (C) a true
diffraction-based reference built from `VectorialPSF`'s established vectorial Debye/
Richards-Wolf optics. All
three run across the same photon-count range used by this repo's other simulation notebooks
(e.g. `Figure_1c.ipynb`, `FigureSI_SBR.ipynb`). The headline result is whether the production
model (A) stays close to the true optics reference (C).

**`FAST_MODE` flag (below):** `True` (default) runs a small, fast grid (fewer photon levels,
300 repeats/condition, fewer wavelength bins for model C) that reproduces the same trend in
well under a minute or two. `FAST_MODE = False` reproduces the full published comparison
(`N_REPEATS=20000`) — hours of compute, since models (B) and (C) are substantially slower than
(A) per condition, and every repeat is also put through the real Gaussian fit.

**Images, not photoelectrons.** The three models differ only in how photons become photoelectrons.
All three then go through the same production camera model (gain, offset, read noise) to give
ADU images, and those images are analysed exactly as real data would be: converted back to
photoelectrons with the camera calibration, then fitted.

**Outputs:** every (dye, photon level, offset) condition is simulated, fitted once and saved as
its own `.npz` (ADU image stacks, full fit results, per-repeat metrics) under `DATA_ROOT` on the lab
backup drive, with the summary CSVs and figures alongside — several GB in full mode, so not in
the repo. A run resumes where it stopped: conditions already on disk are skipped, and files written by an
older version of this notebook (`SIM_VERSION`) stop the run rather than being mixed in.

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(name)s — %(levelname)s — %(message)s',
)

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# Bare `pyS3M.X` imports -- see notebooks/figures/Figure_1c.ipynb for why (Enum/dataclass
# identity consistency with the fitting pipeline's own imports).
from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions, PlottingBase
from pyS3M.PSFFunctions import PSF_Functions
from pyS3M.simulation.multicolour import MultiC_Sim_Funcs
from pyS3M.ImageAnalysisFunctions import Image_Analysis_Functions, FittingStrategy as IAF_FittingStrategy
from pyS3M.sCMOSFunctions import sCMOS_Functions

IO = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
PSF_F = PSF_Functions()
MSF = MultiC_Sim_Funcs()
scmos = sCMOS_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

REPO_ROOT = Path.cwd().resolve().parents[2]  # notebooks/figures/SI -> repo root
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"

# Model (B) costs scale with N_REPEATS x n_photons (each photon gets its own array entry),
# not just N_REPEATS -- much steeper than the other FAST_MODE sweeps in this repo, so both
# axes are cut for FAST_MODE, not just the bootstrap count.
FAST_MODE = False  # False -> the full N_REPEATS=20000 published comparison
print(f"FAST_MODE={FAST_MODE}")

# Everything this notebook produces -- per-condition image stacks, fit results and per-repeat
# metrics, the summary CSVs and the figures -- goes to the lab backup drive: the full run is
# several GB. Fast and full runs get their own subfolders, so a quick check can never
# overwrite (or be mistaken for) the published run.
DATA_ROOT = Path("/media/u2473660/JSBLeeLabBackup/Data/20261002_AvsB")
OUT_ROOT = DATA_ROOT / ("fast" if FAST_MODE else "full")
COND_DIR = OUT_ROOT / "conditions"   # one .npz per (dye, photon level, offset)
FIGURE_DIR = OUT_ROOT / "figures"
for d in (COND_DIR, FIGURE_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"outputs -> {OUT_ROOT}")

In [ ]:
dyes = ['ATTO 488', 'ATTO 565', 'ATTO 647N']

N_REPEATS = 300 if FAST_MODE else 20000
N_PHOTON_STEPS = 8 if FAST_MODE else 15

# Same 500-50,000 log-spaced range/formula as this repo's other simulation-sweep notebooks
# (e.g. FigureSI_SBR.ipynb), not a bespoke set of levels.
photon_levels = np.unique(np.around(np.logspace(np.log10(500), np.log10(50000), N_PHOTON_STEPS) / 5) * 5)

# Sub-pixel offsets from the ROI centre, as a fraction of one pixel -- diagonal-only subset
# of the full Bayer unit cell (see plan §3/§6)
offsets_frac = [(0.0, 0.0), (0.25, 0.25), (0.5, 0.5)]

# Mean background photons reaching each pixel per frame (Poisson, the same for every pixel),
# before detection -- production's definition (SI Eqs. pe_generation, qe_background), used
# identically by all three models below. Each pixel detects them with its background QE: its QE
# curve averaged over a flat spectrum, independent of the dye. BACKGROUND_COLOUR = [1, 1, 1] is
# that flat spectrum (other weights mimic a non-flat one).
BACKGROUND_PHOTONS = 5.0
BACKGROUND_COLOUR = np.array([1.0, 1.0, 1.0])
NA = 1.49
PIXEL_SIZE_NM = 69.0        # Ximea
n_unit_cells = 7            # -> 14x14 px ROI, same convention as Figure_1c.ipynb

RNG_SEED = 12345

# Models (B)/(C) build per-photon arrays of shape (repeats x photons): at 20,000 x 50,000 that is
# 1e9 elements, ~8 GB per float64 array, with several alive at once. Simulating BATCH_REPEATS
# repeats at a time keeps peak memory to a few GB without changing the statistics.
BATCH_REPEATS = 500


def batch_sizes(n_repeats, batch=BATCH_REPEATS):
    '''Split n_repeats into consecutive batches of at most `batch`.'''
    return [min(batch, n_repeats - start) for start in range(0, n_repeats, batch)]

print(f"{len(dyes)} dyes x {len(photon_levels)} photon levels x {len(offsets_frac)} offsets "
      f"x N_REPEATS={N_REPEATS} = {len(dyes)*len(photon_levels)*len(offsets_frac)*N_REPEATS} "
      f"draws per model")

## Camera calibration & Bayer layout

Identical setup to `Figure_1c.ipynb` -- same calibration files, same `MaskFunctions.get_masks`
call, same `pixel_order_indices` convention. Both model (A) and model (B) below read the Bayer
colour of a given pixel from this *same* `masks`/`pixel_order_indices` pair, so the two models
can't disagree because of two different Bayer-layout definitions -- only because of the
factorisation question itself.

In [ ]:
gain = IO.read_tiff(str(CALIBRATION_DIR / "gain.tif"))
offset = IO.read_tiff(str(CALIBRATION_DIR / "offset.tif"))
rqe = IO.read_tiff(str(CALIBRATION_DIR / "rqe.tif"))
variance = IO.read_tiff(str(CALIBRATION_DIR / "variance.tif"))

gain_median = float(np.median(gain))
offset_median = float(np.median(offset))
rqe_median = float(np.median(rqe))
variance_median = float(np.median(variance))          # read-noise variance, ADU^2
readnoise_median = np.sqrt(variance_median) / gain_median  # same quantity in photoelectrons RMS (= readnoise.tif)

R_qe, G_qe, B_qe, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B_qe, G_qe, R_qe])  # B=0, G=1, R=2 -- matches pixel_order below
pixel_order = ['B', 'G', 'R']
pixel_order_indices = {'B': 0, 'G': 1, 'R': 2}
qe_by_channel = [B_qe, G_qe, R_qe]  # same order as pixel_order_indices

H = W = 2 * n_unit_cells
masks = M_F.get_masks(size_x=H, size_y=W)

# Single integer colour-index grid, shared by both models (see markdown above)
colour_grid = np.full((H, W), -1, dtype=int)
for c, idx in pixel_order_indices.items():
    colour_grid[masks[c]] = idx

camera_parameters_base = {
    'gain':                np.full((H, W), gain_median),
    'offset':              np.full((H, W), offset_median),
    'variance':            np.full((H, W), variance_median),
    'readnoise':           readnoise_median,
    'rqe':                 np.full((H, W), rqe_median),
    'masks':               masks,
    'pixel_QYs':           pixel_QYs,
    'pixel_order':         pixel_order,
    'pixel_order_indices': pixel_order_indices,
    'mosaic_unit':         M_F.mosaic_unit,
}

x0_centre_nm = PIXEL_SIZE_NM * H / 2.0
y0_centre_nm = PIXEL_SIZE_NM * W / 2.0
print(f"ROI: {H}x{W} px, centre at ({x0_centre_nm:.1f}, {y0_centre_nm:.1f}) nm")
print(f"camera (calibration medians): gain {gain_median:.4f} ADU/e-, offset {offset_median:.2f} ADU, "
      f"read noise {np.sqrt(variance_median):.3f} ADU = {readnoise_median:.3f} e- RMS")

## Shared helpers: background rate, camera model, the Gaussian fit, and the spectral fingerprint

`background_photoelectrons` gives models (B)/(C) exactly the background production model (A) uses:
$\mathrm{Binomial}(\mathrm{Poisson}(b_\mathrm{photons}), \mathrm{QE}_{\mathrm{bg},n})$, with $\mathrm{QE}_{\mathrm{bg},n}$ from production's
`MultiC_Sim_Funcs.background_pixel_efficiency` (flat background spectrum, independent of the dye).

**Camera and analysis, shared by all three models.** Each model produces photoelectrons per pixel;
`camera_image` turns those into an ADU image with production's own camera model
(`PSF_Functions.photoelectrons_to_image`, the function `gen_camera_image_stack` calls):
$w_n = \mathrm{Normal}(\gamma_n N_{\mathrm{pe},n} + \mu_n, \sigma^2_{\mathrm{ro},n})$, clipped at 0, using the
Ximea calibration medians. `fit_stack` then analyses that ADU image the way the pipeline
analyses real data: `to_photoelectrons` converts it back with `IO_Functions.convert_to_photoelectrons`,
$(w_n - \mu_n)/\gamma_n/\mathrm{rqe}_n$, before smoothing, weighting and fitting.
`fit_stack` runs a real nonlinear, colour-aware 2D Gaussian fit on every repeat
-- `ImageAnalysisFunctions.Image_Analysis_Functions.fit_puncta_method` with the `STANDARD_DATA`
strategy (three re-weighted least-squares passes, the last using raw-data weights to remove a
high-photon-count amplitude bias), the same strategy this repo's other `FigureSI_*.ipynb`
simulation notebooks use for their own reported precision numbers. Colour-aware because these
images are raw Bayer-mosaiced (one channel's counts per pixel) -- `NOCOLOUR` is only ever
applied to a *demosaiced* grayscale image in production (`multicolour.py::_fit_demosaic_ig`),
so it would be a model mismatch here.

**Requires the background-guess fix (`gaussoptfuncs._BG_GUESS_FLOOR`, 2026-10-02).** Before it,
any channel whose minimum pixel was 0 -- most repeats at a few background photons per pixel --
started the fit with a background of exactly 0, and the fit stalled with all three colour
amplitudes stuck at their equal starting guess. That corrupted the fingerprint *and* sigma_PSF
(in this notebook's own conditions, 46-71% of fits). Results from before the fix are not
comparable. The run cell below counts fits whose amplitudes come back exactly equal, saves that
fraction with each condition, warns on any, and stops the run above `MAX_STUCK_FRACTION` (0.5%),
so a regression cannot pass silently.

Two quantities come out of that one fit:

- **sigma_PSF**: the isotropic width $(\sigma_x+\sigma_y)/2$.
- **spectral fingerprint**: the fitted per-channel amplitudes normalised to sum to one,
  $(A_B, A_G, A_R)/\sum A$ -- pyS3M's own colour convention (`IO_Functions._normalize_color_channels`).
  The fit carries its own background per channel, so unlike raw per-channel pixel sums this is
  not diluted by background, and it uses all three channels rather than picking one.

**Ground truth for the fingerprint.** These simulations detect photons by Binomial thinning with
each pixel's *absolute* QE, so the fitted amplitudes estimate $A_c \propto \int p(\lambda)QE_c(\lambda)d\lambda$
and the true fingerprint is that ratio of integrals, normalised: `get_pixel_fractions_dye_and_filters(normalized=False)`
divided by its sum. **Not** `normalized=True`, which is $\int p(\lambda)\,QE_c/\sum_{c'}QE_{c'}\,d\lambda$ --
the right truth for pyS3M's stochastic photon-to-channel colour assignment, but a different quantity
here (0.009 away for ATTO 488, ~0.003 for the red dyes -- comparable to the fingerprint's own
noise at high photon counts, so the wrong choice would put a false floor under the distance).
The **fingerprint distance** is the Euclidean distance between fitted and true fingerprints,
per repeat. It is non-negative, so even an unbiased model has a mean distance above zero,
falling roughly as $1/\sqrt{N}$ -- it measures total colour error (bias + noise), compared
between models at equal photon count.

In [ ]:
# Background photoelectrons, exactly as production model (A) draws them
# (multicolour.py::gen_camera_image_stack; SI Eqs. pe_generation, qe_background): every pixel
# receives Poisson(BACKGROUND_PHOTONS) photons, scaled by its relative QE (rqe) like the signal,
# each detected with the pixel's background QE -- its QE curve averaged over a flat spectrum,
# from production's own background_pixel_efficiency. Models (B)/(C) draw from this, so all
# three models see the same background.
background_qe_per_colour = MSF.background_pixel_efficiency(wavelength, pixel_QYs, BACKGROUND_COLOUR)
background_qe_grid = np.zeros((H, W))
for c, idx in pixel_order_indices.items():
    background_qe_grid[masks[c]] = background_qe_per_colour[idx]


def background_photoelectrons(n_repeats, camera_parameters, rng):
    '''Background photoelectrons, (n_repeats, H, W): Binomial(Poisson(b * rqe), QE_bg,n).'''
    photons = rng.poisson(BACKGROUND_PHOTONS * camera_parameters['rqe'], size=(n_repeats, H, W))
    return rng.binomial(photons, background_qe_grid)


print(f"background: {BACKGROUND_PHOTONS:g} photons/pixel -> mean detected photoelectrons/pixel "
      + "  ".join(f"{c}={BACKGROUND_PHOTONS * background_qe_per_colour[i]:.2f}" for c, i in pixel_order_indices.items()))


def camera_image(pe_stack, camera_parameters):
    '''Photoelectrons -> ADU image stack, with production's camera model
    (PSF_Functions.photoelectrons_to_image, as called by gen_camera_image_stack):
    Normal(gain * pe + offset, variance), clipped at 0. Same function for all three models.'''
    return np.stack([
        PSF_F.photoelectrons_to_image(frame, camera_parameters['gain'], camera_parameters['offset'],
                                      camera_parameters['variance'])
        for frame in pe_stack
    ])


def to_photoelectrons(adu, camera_parameters):
    '''ADU -> photoelectrons as the analysis pipeline does for real data
    (IO_Functions.convert_to_photoelectrons): (ADU - offset) / gain / rqe.'''
    return IO.convert_to_photoelectrons(
        np.asarray(adu, dtype=float), gain_map=camera_parameters['gain'],
        offset_map=camera_parameters['offset'], rqe=camera_parameters['rqe'],
    )


IAF = Image_Analysis_Functions(readnoise=camera_parameters_base['readnoise'])

# Explicit pixel_order stacking, not dict-iteration order: multicolour.py's own
# _build_frame_masks stacks by dict-key order, which only happens to match pixel_order because
# MaskFunctions.get_masks builds its dict via np.unique (alphabetically B/G/R) -- not a
# guaranteed invariant. Stacking explicitly here removes that fragile coincidence.
masks_3d = np.dstack([masks[c] for c in pixel_order])  # (H, W, n_colours), channel i == pixel_order[i]
N_CH = len(pixel_order)


# Real nonlinear, colour-aware 2D Gaussian fit, per repeat -- production's STANDARD_DATA
# strategy (ImageAnalysisFunctions.Image_Analysis_Functions.fit_puncta_method), the same
# strategy this repo's other FigureSI_*.ipynb simulation notebooks use for their own reported
# precision numbers, called stand-alone (no demosaic pre-pass needed -- confirmed against
# multicolour.py::_fit_standard_data, the production call site). Takes the ADU image stack and
# converts it to photoelectrons first, as the pipeline does. Background is fit as a free
# parameter per channel (no separate background subtraction needed). Returns the full fit
# array, one row per repeat: [x, y, sx, sy, bg_0..bg_{n-1}, A_0..A_{n-1}, ...] with channels in
# pixel_order (masks_3d's stacking order). Rows where the fit's own in-bounds rejection gate
# discards the repeat are NaN -- callers should drop NaNs before aggregating.
def fit_stack(adu_stack):
    data = to_photoelectrons(adu_stack, camera_parameters_base)  # as for real data
    smoothed = scmos.gaussian_filter_stack(data, sigma=1.5)
    weights = 1.0 / (np.clip(smoothed, 0, None) + 1.0 + camera_parameters_base['readnoise'] ** 2)
    n = data.shape[0]
    masks_tofit = [masks_3d] * n  # same fixed Bayer pattern for every repeat

    fit, _ = IAF.fit_puncta_method(
        list(data), list(smoothed), list(weights),
        [(0.0, 0.0)] * n, list(range(n)),
        strategy=IAF_FittingStrategy.STANDARD_DATA,
        masks=masks_tofit,
    )
    return fit


def fit_sigma_nm(fit):
    '''Isotropic sigma_PSF in nm, (sigma_x + sigma_y) / 2.'''
    return 0.5 * (fit[:, 2] + fit[:, 3]) * PIXEL_SIZE_NM


def fit_fingerprint(fit):
    '''Fitted spectral fingerprint per repeat: (A_B, A_G, A_R) / sum(A), in pixel_order.
    NaN where the fit failed or the amplitudes sum to <= 0.'''
    amps = fit[:, 4 + N_CH:4 + 2 * N_CH]
    total = amps.sum(axis=1, keepdims=True)
    return np.where(total > 0, amps / np.where(total > 0, total, 1.0), np.nan)


def stuck_fraction(fit):
    '''Fraction of successful fits whose three colour amplitudes are exactly equal -- the
    signature of a fit that stalled at its starting guess (see markdown above).'''
    amps = fit[:, 4 + N_CH:4 + 2 * N_CH]
    ok = ~np.isnan(amps).any(axis=1)
    stuck = ok & np.isclose(amps, amps[:, :1]).all(axis=1)
    return stuck.sum() / max(ok.sum(), 1)


def true_fingerprint(dye):
    '''Ground-truth fingerprint (pixel_order): normalised ratio of integrals
    int p(lambda) QE_c(lambda) dlambda -- what the fitted amplitudes estimate here, since
    detection is Binomial thinning with each pixel's absolute QE (see markdown above).'''
    _, abs_qe = S_F.get_pixel_fractions_dye_and_filters(
        [dye], [], wavelength, pixel_QYs, normalized=False,
    )
    abs_qe = np.ravel(abs_qe)
    return abs_qe / abs_qe.sum()


for dye in dyes:
    print(f"{dye}: true fingerprint "
          + "  ".join(f"{c}={v:.4f}" for c, v in zip(pixel_order, true_fingerprint(dye))))

## Model (A) — factorised, the real production code

Not reimplemented: the same calls `MultiC_Sim_Funcs.test_simulation_method` makes for every other
simulation in the paper, in its stochastic photon mode (`use_stochastic_photons=True`, the mode
all the paper's simulations use). For each repeat, `Spectral_Funcs.generate_bootstrap_colour_ratios`
samples `n_photons` wavelengths from the dye spectrum, giving that repeat's own mean wavelength
(hence PSF width), and returns the per-channel detection efficiencies $\mathrm{QE}_{\mathrm{d},c} =
\int \mathrm{PDF}_\mathrm{Spectrum}\,\mathrm{QE}_c\,d\lambda$ (the SI's equation, the same for every repeat; the
image's Poisson/Binomial steps supply the colour shot noise). `gen_camera_image_stack` then renders
the image. It is factorised -- within a repeat every photon shares one PSF width and one set of
detection efficiencies -- unlike model (B), where each photon's own wavelength sets both.
(Before 2026-10-05 the colour ratios were drawn from the sampled photons, which biased green
dyes by up to ~0.8 pp and double-counted colour noise; SIM_VERSION 5 uses the fixed code.)
`return_photoelectrons_stack=True` gives photoelectrons per pixel straight after the Binomial
QE step (Eq. S8). The camera step (gain, offset, read noise) is then applied by the shared
`camera_image`, the same production function `gen_camera_image_stack` would have used, so
all three models get an identical camera. Simulated `BATCH_REPEATS` repeats at a time, since the
photon sampling builds a (repeats x photons) array.

In [ ]:
def _simulate_factorised_batch(dye, n_photons_val, x0_nm, y0_nm, n_repeats, camera_parameters, rng):
    '''Model (A) for one batch of repeats: production's stochastic photon mode, exactly as
    MultiC_Sim_Funcs.test_simulation_method calls it (use_stochastic_photons=True).'''
    dye_spectrum = S_F.get_dye_or_filter_data(names=[dye], wavelength=wavelength, dye_or_filter=True)[0]
    # Per repeat: n_photons sampled wavelengths -> that repeat's mean wavelength (PSF width);
    # colour ratios are the deterministic per-channel detection efficiencies
    mean_wavelengths, colour_ratios = S_F.generate_bootstrap_colour_ratios(
        dye_spectrum, wavelength, camera_parameters['pixel_QYs'],
        n_photons_per_image=int(n_photons_val), n_bootstrap=n_repeats,
        pixel_order=camera_parameters['pixel_order'],
        pixel_order_indices=camera_parameters['pixel_order_indices'],
        random_state=rng,
    )
    x0y0 = {"dye": np.zeros([n_repeats, 2, 1])}
    x0y0["dye"][:, 0, 0] = y0_nm  # row axis = y
    x0y0["dye"][:, 1, 0] = x0_nm  # col axis = x
    n_photons = {"dye": np.full(n_repeats, n_photons_val, dtype=float)}

    return MSF.gen_camera_image_stack(
        camera_parameters, wavelength, mean_wavelengths, colour_ratios, n_photons, x0y0,
        smoothing_function=None,
        background_photons=BACKGROUND_PHOTONS,
        background_colour=list(BACKGROUND_COLOUR),
        NA=NA, pixel_size=PIXEL_SIZE_NM,
        return_photoelectrons_stack=True,
    )  # (n_repeats, H, W) int32 photoelectrons


def simulate_factorised(dye, n_photons_val, x0_nm, y0_nm, n_repeats, camera_parameters, rng):
    '''Model (A): the real, production factorised simulation path (stochastic photon mode), in
    batches of BATCH_REPEATS repeats so the (repeats x photons) wavelength samples fit in memory.'''
    return np.concatenate([
        _simulate_factorised_batch(dye, n_photons_val, x0_nm, y0_nm, n, camera_parameters, rng)
        for n in batch_sizes(n_repeats)
    ])

## Model (B) — non-factorised, photon-by-photon reference

New implementation. For each photon: its own sampled
wavelength gives *both* its own PSF width (spatial placement) *and* its own per-channel
detection probability (no spectrum-averaging of either). The number of photons emitted per
repeat is drawn as Poisson(`n_photons`), matching model (A), whose per-pixel Poisson($I_n$)
draws sum to Poisson(`n_photons`) -- so the two models share the same photon-count noise. Fully vectorised over
(repeat, photon) -- no Python-level loop over either axis, so this should already be about as
fast as a pure-numpy implementation gets; if it's still too slow at scale the next step would
be numba, not restructuring this. Simulated `BATCH_REPEATS` repeats at a time to bound memory
(see the setup cell).

In [ ]:
def _sample_wavelengths(spectrum, wl, shape, rng):
    '''Vectorised inverse-CDF sampling -- same method as
    SpectralFunctions.sample_photons_from_spectrum, generalised to an arbitrary output shape.'''
    pdf = np.maximum(spectrum, 0)
    pdf = pdf / np.trapz(pdf, wl)
    dx = np.diff(wl)
    pdf_avg = 0.5 * (pdf[1:] + pdf[:-1])
    cdf = np.zeros(len(wl))
    cdf[1:] = np.cumsum(pdf_avg * dx)
    cdf = cdf / cdf[-1]
    u = rng.random(shape)
    return np.interp(u.ravel(), cdf, wl).reshape(shape)


def _simulate_nonfactorised_batch(dye, n_photons_val, x0_nm, y0_nm, n_repeats, camera_parameters, rng):
    '''Model (B) for one batch of repeats: joint wavelength-resolved reference, photon-by-photon.'''
    dye_spectrum = S_F.get_dye_or_filter_data(names=[dye], wavelength=wavelength, dye_or_filter=True)[0]

    # Photons emitted per repeat ~ Poisson(n_photons), like model (A): its per-pixel Poisson(I_n)
    # draws sum to Poisson(n_photons). Arrays are sized to the largest draw in the batch, and
    # photons beyond a repeat's own draw are masked out below.
    n_emitted = rng.poisson(n_photons_val, size=n_repeats)
    shape = (n_repeats, int(n_emitted.max()))
    emitted = np.arange(shape[1])[None, :] < n_emitted[:, None]
    lam = _sample_wavelengths(dye_spectrum, wavelength, shape, rng)          # own wavelength...
    sigma_px = PSF_F.sigma_PSF(lam, NA) / PIXEL_SIZE_NM                       # ...own PSF width

    x0_px, y0_px = x0_nm / PIXEL_SIZE_NM, y0_nm / PIXEL_SIZE_NM
    xk = x0_px + rng.standard_normal(shape) * sigma_px
    yk = y0_px + rng.standard_normal(shape) * sigma_px
    ik = np.round(xk).astype(np.int64)  # column index
    jk = np.round(yk).astype(np.int64)  # row index

    H, W = camera_parameters['gain'].shape
    valid = (ik >= 0) & (ik < W) & (jk >= 0) & (jk < H)
    colour_k = np.full(shape, -1, dtype=np.int64)
    colour_k[valid] = colour_grid[jk[valid], ik[valid]]

    QE_k = np.zeros(shape)                                                    # ...own detection prob.
    for c_idx, curve in enumerate(qe_by_channel):
        sel = colour_k == c_idx
        if np.any(sel):
            QE_k[sel] = np.interp(lam[sel], wavelength, curve)

    # Production scales each pixel's detection probability by its relative QE (rqe), so do the same.
    QE_k[valid] *= camera_parameters['rqe'][jk[valid], ik[valid]]
    detected = emitted & valid & (rng.random(shape) < QE_k)

    r_idx = np.broadcast_to(np.arange(n_repeats)[:, None], shape)
    flat_idx = (r_idx * H + jk) * W + ik
    counts = np.bincount(flat_idx[detected], minlength=n_repeats * H * W)
    image = counts.reshape(n_repeats, H, W).astype(np.int32)

    # Background: production's definition (see background_photoelectrons) -- flat spectrum,
    # detected with each pixel's background QE -- so this model sees exactly model (A)'s background.
    image = image + background_photoelectrons(n_repeats, camera_parameters, rng).astype(np.int32)

    return image


def simulate_nonfactorised(dye, n_photons_val, x0_nm, y0_nm, n_repeats, camera_parameters, rng):
    '''Model (B): joint wavelength-resolved reference, photon-by-photon, in batches of
    BATCH_REPEATS repeats so the (repeats x photons) arrays fit in memory.'''
    return np.concatenate([
        _simulate_nonfactorised_batch(dye, n_photons_val, x0_nm, y0_nm, n, camera_parameters, rng)
        for n in batch_sizes(n_repeats)
    ])

## Model (C) — true diffraction-based reference (VectorialPSF)

Both (A) and (B) above share the same Gaussian PSF-width approximation
(`PSF_Functions.sigma_PSF`, σ = λ/(√2·π·NA)) — (B) only wavelength-resolves *which* σ each
photon gets, it doesn't change the Gaussian functional form itself. This builds a third reference that replaces the Gaussian entirely
with a genuine vectorial Debye/Richards-Wolf diffraction PSF
(`simulation/defocus_psf.py::VectorialPSF` — Fourier-optics propagation of the apodised pupil
field). Established equations, not an approximation of an approximation — and not new,
unvalidated code: this class was already built and unit-tested (16 tests: normalisation,
in-focus symmetry, defocus broadening, spherical-aberration symmetry-breaking, lateral shift,
aperture clipping) for this project's Z-defocus simulation work; this notebook is its first
use as a validation reference for the in-focus factorisation question.

**Computational note**: an FFT-based diffraction patch for one (wavelength, sub-pixel offset)
pair is too expensive to recompute per individual photon at its own exact continuous
wavelength. Instead, each dye's emission spectrum is discretised into `N_WAVELENGTH_BINS`
representative wavelengths (stratified by spectral probability mass, not evenly spaced), and
the diffraction patch is precomputed once per (dye, offset, bin) — reused across every
photon_level/repeat below, since the *shape* of the diffraction pattern only depends on
wavelength/offset, not on how many photons are drawn from it. Each photon's own continuous
wavelength is snapped to its nearest bin for spatial placement only; detection QE still uses
the continuous value, unchanged from model (B).

In [ ]:
from pyS3M.simulation.defocus_psf import VectorialPSF

N_WAVELENGTH_BINS = 8 if FAST_MODE else 15


def build_wavelength_bins(spectrum, wl, n_bins):
    '''n_bins representative wavelengths + bin edges, stratified by the dye's own
    emission-spectrum CDF (equal probability mass per bin, not equal wavelength spacing).'''
    pdf = np.maximum(spectrum, 0)
    pdf = pdf / np.trapz(pdf, wl)
    dx = np.diff(wl)
    pdf_avg = 0.5 * (pdf[1:] + pdf[:-1])
    cdf = np.zeros(len(wl))
    cdf[1:] = np.cumsum(pdf_avg * dx)
    cdf = cdf / cdf[-1]
    edges_p = np.linspace(0, 1, n_bins + 1)
    edges_wl = np.interp(edges_p, cdf, wl)
    centres_p = (edges_p[:-1] + edges_p[1:]) / 2
    centres_wl = np.interp(centres_p, cdf, wl)
    return centres_wl, edges_wl


vpsf = VectorialPSF(NA=NA, n_medium=1.33, n_immersion=1.515, pix_obj_um=PIXEL_SIZE_NM / 1000.0)
roi_centre_row = int(round(y0_centre_nm / PIXEL_SIZE_NM))
roi_centre_col = int(round(x0_centre_nm / PIXEL_SIZE_NM))

diffraction_cache = {}  # (dye, offset) -> (patches (n_bins, ps, ps), bin_edges_wl)
t0 = time.time()
for dye in dyes:
    dye_spectrum = S_F.get_dye_or_filter_data(names=[dye], wavelength=wavelength, dye_or_filter=True)[0]
    bin_centres_wl, bin_edges_wl = build_wavelength_bins(dye_spectrum, wavelength, N_WAVELENGTH_BINS)
    for (fx, fy) in offsets_frac:
        patches = vpsf.compute_psf_stack(
            z_offsets_um=np.array([0.0]),
            wavelengths_um=bin_centres_wl / 1000.0,
            x0_um=fx * (PIXEL_SIZE_NM / 1000.0),
            y0_um=fy * (PIXEL_SIZE_NM / 1000.0),
        )[0]  # (N_WAVELENGTH_BINS, psf_size, psf_size)
        diffraction_cache[(dye, (fx, fy))] = (patches, bin_edges_wl)

print(f"Precomputed {len(diffraction_cache)} (dye, offset) diffraction PSF sets "
      f"({N_WAVELENGTH_BINS} wavelength bins each, psf_size={vpsf.psf_size}) "
      f"in {time.time()-t0:.1f}s")

In [ ]:
def _simulate_diffraction_batch(dye, n_photons_val, n_repeats, camera_parameters, rng, offset):
    '''Model (C) for one batch of repeats: true vectorial-diffraction reference, photon-by-photon. Spatial placement
    draws from a precomputed VectorialPSF patch (nearest wavelength bin); detection QE uses
    each photon's own continuous sampled wavelength, unchanged from model (B).'''
    n_photons_int = int(n_photons_val)
    dye_spectrum = S_F.get_dye_or_filter_data(names=[dye], wavelength=wavelength, dye_or_filter=True)[0]
    patches, bin_edges_wl = diffraction_cache[(dye, offset)]

    shape = (n_repeats, n_photons_int)
    lam = _sample_wavelengths(dye_spectrum, wavelength, shape, rng)
    bin_idx = np.clip(np.digitize(lam, bin_edges_wl) - 1, 0, len(patches) - 1)

    H, W = camera_parameters['gain'].shape
    ps = patches.shape[1]
    patch_centre = ps // 2

    ik = np.zeros(shape, dtype=np.int64)
    jk = np.zeros(shape, dtype=np.int64)
    for b in range(len(patches)):
        sel = bin_idx == b
        n_sel = int(sel.sum())
        if n_sel == 0:
            continue
        flat_p = patches[b].ravel()
        flat_p = flat_p / flat_p.sum()
        draw = rng.choice(flat_p.size, size=n_sel, p=flat_p)
        r, c = np.unravel_index(draw, patches[b].shape)
        jk[sel] = roi_centre_row - patch_centre + r
        ik[sel] = roi_centre_col - patch_centre + c

    valid = (ik >= 0) & (ik < W) & (jk >= 0) & (jk < H)
    colour_k = np.full(shape, -1, dtype=np.int64)
    colour_k[valid] = colour_grid[jk[valid], ik[valid]]

    QE_k = np.zeros(shape)
    for c_idx, curve in enumerate(qe_by_channel):
        sel = colour_k == c_idx
        if np.any(sel):
            QE_k[sel] = np.interp(lam[sel], wavelength, curve)

    # Production scales each pixel's detection probability by its relative QE (rqe), so do the same.
    QE_k[valid] *= camera_parameters['rqe'][jk[valid], ik[valid]]
    detected = valid & (rng.random(shape) < QE_k)

    r_idx = np.broadcast_to(np.arange(n_repeats)[:, None], shape)
    flat_idx = (r_idx * H + jk) * W + ik
    counts = np.bincount(flat_idx[detected], minlength=n_repeats * H * W)
    image = counts.reshape(n_repeats, H, W).astype(np.int32)

    # Background: production's definition (see background_photoelectrons) -- flat spectrum,
    # detected with each pixel's background QE -- so this model sees exactly model (A)'s background.
    image = image + background_photoelectrons(n_repeats, camera_parameters, rng).astype(np.int32)

    return image


def simulate_diffraction(dye, n_photons_val, n_repeats, camera_parameters, rng, offset):
    '''Model (C): true vectorial-diffraction reference, in batches of BATCH_REPEATS repeats so
    the (repeats x photons) arrays fit in memory.'''
    return np.concatenate([
        _simulate_diffraction_batch(dye, n_photons_val, n, camera_parameters, rng, offset)
        for n in batch_sizes(n_repeats)
    ])

## Run all three models, fit once, save per condition

Runs (A) factorised, (B) photon-by-photon (still Gaussian), and (C) the true diffraction-based
reference (`VectorialPSF`) at the `N_REPEATS`/`photon_levels` set above across every (dye,
photon level, offset) condition. Each model's photoelectrons go through the shared camera model
to ADU images, which are fitted once, right away, and everything -- ADU images (float32), full
fit results, and the per-repeat metrics used below (`N` photoelectrons in the analysed image, `sigma` PSF width, `fp` fitted fingerprint, `fpdist` its distance
from the truth) -- is written to one `.npz` in `COND_DIR`. Only that file is kept, nothing is
held in memory, and every later cell reads from it rather than refitting.

**Resumable:** a condition whose file already exists is skipped (and one saved by an older
`SIM_VERSION` stops the run, so old and new results are never mixed), and files are written under a
temporary name and renamed on completion, so an interrupted run leaves no half-written
results. **Seeding:** each model draws from its own generator seeded per condition, so results do not
depend on run order or resumption. The production code (model A's image rendering and the shared
camera step) also draws from NumPy's global RNG (seeded per condition here) and from numba's own
internal RNG (not seedable from Python), so runs are not bit-for-bit reproducible.

In [ ]:
def condition_path(dye, n_photon, offset):
    fx, fy = offset
    return COND_DIR / f"{dye.replace(' ', '-')}_n{int(n_photon)}_off{fx:g}-{fy:g}.npz"


def load_condition(dye, n_photon, offset, keys=None):
    '''One condition's saved results; `keys` loads only those arrays (npz members load lazily).'''
    with np.load(condition_path(dye, n_photon, offset)) as z:
        return {k: z[k] for k in (z.files if keys is None else keys)}


MODELS = ('A', 'B', 'C')
# Bump whenever the simulation or analysis changes, so stale condition files are never reused.
# 2: model (B) Poisson photon count; all models simulate ADU images (calibrated camera) and are
#    analysed from them.
# 3: model (A) in production's stochastic photon mode.
# 4: background photons detected with each pixel's flat-spectrum background QE (was QE = 1).
# 5: stochastic photon mode's colour ratios fixed (deterministic detection efficiencies; were
#    biased by up to ~0.8 pp for green dyes and double-counted colour noise).
SIM_VERSION = 5
# Stop the run if more fits than this stall at equal colour amplitudes (the pre-fix bug did 46-71%).
MAX_STUCK_FRACTION = 0.005
t_start = time.time()
n_new = n_skipped = 0
for i_dye, dye in enumerate(dyes):
    truth = true_fingerprint(dye)
    for i_level, n_photon in enumerate(photon_levels):
        for i_off, (fx, fy) in enumerate(offsets_frac):
            path = condition_path(dye, n_photon, (fx, fy))
            if path.exists():
                with np.load(path) as z:
                    saved = int(z['sim_version']) if 'sim_version' in z.files else 1
                if saved != SIM_VERSION:
                    raise RuntimeError(
                        f"{path} was saved by SIM_VERSION {saved}, not {SIM_VERSION}: move or delete "
                        f"{COND_DIR} before re-running")
                n_skipped += 1
                continue
            t_cond = time.time()
            x0_nm = x0_centre_nm + fx * PIXEL_SIZE_NM
            y0_nm = y0_centre_nm + fy * PIXEL_SIZE_NM
            seed = [RNG_SEED, i_dye, i_level, i_off]

            np.random.seed(seed)  # production code (model A, camera step) also draws from the global RNG
            pe_stacks = {
                'A': simulate_factorised(dye, n_photon, x0_nm, y0_nm, N_REPEATS, camera_parameters_base,
                                         np.random.default_rng(seed + [0])),
                'B': simulate_nonfactorised(dye, n_photon, x0_nm, y0_nm, N_REPEATS, camera_parameters_base,
                                            np.random.default_rng(seed + [1])),
                'C': simulate_diffraction(dye, n_photon, N_REPEATS, camera_parameters_base,
                                          np.random.default_rng(seed + [2]), (fx, fy)),
            }

            out = {'truth': truth, 'sim_version': SIM_VERSION}
            for m, pe in pe_stacks.items():
                img = camera_image(pe, camera_parameters_base)  # identical camera for every model
                fit = fit_stack(img)
                stuck = stuck_fraction(fit)
                if stuck > MAX_STUCK_FRACTION:
                    raise RuntimeError(
                        f"{path.name} model {m}: {100 * stuck:.1f}% of fits stalled with equal colour "
                        "amplitudes -- pyS3M's background-guess fix (gaussoptfuncs._BG_GUESS_FLOOR) is "
                        "missing or has regressed")
                if stuck > 0:
                    print(f"  WARNING {path.name} model {m}: {100 * stuck:.2f}% of fits stalled")
                out[f'stuck_{m}'] = stuck
                fp = fit_fingerprint(fit)
                out[f'img_{m}'] = img.astype(np.float32)  # ADU
                out[f'fit_{m}'] = fit
                out[f'N_{m}'] = to_photoelectrons(img, camera_parameters_base).sum(axis=(1, 2))
                out[f'sigma_{m}'] = fit_sigma_nm(fit)
                out[f'fp_{m}'] = fp
                out[f'fpdist_{m}'] = np.linalg.norm(fp - truth, axis=1)

            tmp = path.with_name(path.stem + '.tmp.npz')
            np.savez(tmp, **out)
            tmp.replace(path)
            n_new += 1
            print(f"  {path.name}: {time.time() - t_cond:.1f}s", flush=True)

total_time = time.time() - t_start
n_conditions = len(dyes) * len(photon_levels) * len(offsets_frac)
print(f"{n_conditions} conditions x N_REPEATS={N_REPEATS}: {n_new} run, {n_skipped} already on disk; "
      f"wall time {total_time/60:.1f} min")

## Visual comparison

Mean image over the `N_REPEATS` repeats for (A) factorised, (B) photon-by-photon (still
Gaussian), and (C) the true diffraction reference, for one representative condition per dye
(middle photon level, zero offset). The right-most panel is (A) − (C) — the actual quantity of
interest: how far the production model departs from the true optics.

In [ ]:
mid_photon = photon_levels[len(photon_levels) // 2]
fig, axs = plotter.two_column_plot(nrows=len(dyes), ncols=4, width=6.69, height=6.0)

for row, dye in enumerate(dyes):
    imgs = load_condition(dye, mid_photon, (0.0, 0.0), keys=['img_A', 'img_B', 'img_C'])
    mean_A = to_photoelectrons(imgs['img_A'].mean(axis=0), camera_parameters_base)
    mean_B = to_photoelectrons(imgs['img_B'].mean(axis=0), camera_parameters_base)
    mean_C = to_photoelectrons(imgs['img_C'].mean(axis=0), camera_parameters_base)
    diff = mean_A - mean_C
    vmax = max(mean_A.max(), mean_B.max(), mean_C.max())

    for col, (img, title, cmap, vmin, vm) in enumerate([
        (mean_A, f'{dye}\n(A) factorised', 'gist_gray', 0, vmax),
        (mean_B, '(B) photon-by-photon\n(Gaussian)', 'gist_gray', 0, vmax),
        (mean_C, '(C) diffraction\n(VectorialPSF)', 'gist_gray', 0, vmax),
        (diff, 'A - C', 'RdBu_r', -np.abs(diff).max(), np.abs(diff).max()),
    ]):
        ax = axs[row, col]
        im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vm)
        ax.set_title(title, fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, pad=0.02, fraction=0.046)

plt.savefig(FIGURE_DIR / "visual_comparison.svg", format='svg', dpi=600)
plt.show()

In [ ]:
mid_photon = photon_levels[len(photon_levels) // 2]
fig, axs = plotter.two_column_plot(nrows=len(dyes), ncols=3)

for row, dye in enumerate(dyes):
    imgs = load_condition(dye, mid_photon, (0.0, 0.0), keys=['img_A', 'img_B', 'img_C'])
    mean_A = to_photoelectrons(imgs['img_A'].mean(axis=0), camera_parameters_base)
    mean_B = to_photoelectrons(imgs['img_B'].mean(axis=0), camera_parameters_base)
    diff = mean_A - mean_B
    vmax = max(mean_A.max(), mean_B.max())

    for col, (img, title, cmap, vmin, vm) in enumerate([
        (mean_A, f'{dye}\n(A) factorised', 'gist_gray', 0, vmax),
        (mean_B, '(B) photon-by-photon\n(Gaussian)', 'gist_gray', 0, vmax),
        (diff, 'A - B', 'RdBu_r', -np.abs(diff).max(), np.abs(diff).max()),
    ]):
        ax = axs[row, col]
        im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vm)
        ax.set_title(title, fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, pad=0.02, fraction=0.046)

plt.savefig(FIGURE_DIR / "visual_comparison_avsb.svg")
plt.show()

## Quantitative comparison

First, per-condition aggregate summary (every dye x photon-level x offset cell), framed against
(C) the true diffraction reference — then a closer look at three key differentiators
(sigma_PSF, N_photoelectrons, spectral-fingerprint distance) as full distributions, not just means,
below.

In [ ]:
# Per-condition means of the saved per-repeat metrics. Fingerprint columns are the fitted
# (background-free) per-channel fractions, compared against the true fingerprint in percentage
# points; fpdist is the mean per-repeat distance from the truth.
SUMMARY_KEYS = [f'{q}_{m}' for q in ('N', 'fp', 'fpdist') for m in MODELS]

rows = []
for dye in dyes:
    truth = true_fingerprint(dye)
    for n_photon in photon_levels:
        for (fx, fy) in offsets_frac:
            c = load_condition(dye, n_photon, (fx, fy), keys=SUMMARY_KEYS)
            row = {'dye': dye, 'n_photon': n_photon, 'offset': f'({fx},{fy})'}
            fp_mean = {}
            for m in MODELS:
                row[f'total_{m}_mean'] = c[f'N_{m}'].mean()
                row[f'fpdist_{m}_mean'] = np.nanmean(c[f'fpdist_{m}'])
                fp_mean[m] = np.nanmean(c[f'fp_{m}'], axis=0)
            for i, ch in enumerate(pixel_order):
                row[f'fp_{ch}_true'] = truth[i]
                for m in MODELS:
                    row[f'fp_{ch}_{m}'] = fp_mean[m][i]
                row[f'fp_{ch}_diff_AvsB_pp'] = (fp_mean['A'][i] - fp_mean['B'][i]) * 100
                row[f'fp_{ch}_diff_AvsC_pp'] = (fp_mean['A'][i] - fp_mean['C'][i]) * 100
                row[f'fp_{ch}_diff_BvsC_pp'] = (fp_mean['B'][i] - fp_mean['C'][i]) * 100
            rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUT_ROOT / "factorisation_comparison_summary.csv", index=False)
summary_df[['dye', 'n_photon', 'offset', 'total_A_mean', 'total_B_mean', 'total_C_mean',
            'fpdist_A_mean', 'fpdist_B_mean', 'fpdist_C_mean',
            'fp_R_diff_AvsC_pp', 'fp_G_diff_AvsC_pp', 'fp_B_diff_AvsC_pp']]

### Distributional comparison: sigma_PSF, N_photoelectrons, spectral-fingerprint distance

Pooled across the 3 Bayer-phase offsets, at the middle photon level, per dye (`3 x N_REPEATS`
draws per model per dye) -- enough for a real look at distribution shape, not just means.

- **sigma_PSF**: a real nonlinear, colour-aware 2D Gaussian fit (`fit_stack`, production's `STANDARD_DATA` strategy -- see "Shared helpers" above), the same strategy/quantity this repo's other simulation notebooks report. A small fraction of repeats can fail the fit's own in-bounds rejection gate (mostly at very low photon counts) and come back NaN -- dropped before plotting/statistics; the printed NaN rate above is worth checking per condition rather than assumed to be zero.
- **N_photoelectrons**: total photoelectrons per repeat in the analysed image (ADU converted back to photoelectrons, so including background and read noise), no fitting needed.
- **spectral-fingerprint distance**: per repeat, the Euclidean distance between the fitted fingerprint $(A_B,A_G,A_R)/\sum A$ and the dye's true fingerprint (see "Shared helpers" above). One number per repeat that uses all three channels, free of background dilution, and needing no choice of channel. Repeats whose fit failed are NaN and dropped, as for sigma_PSF.

Each row below is one (dye, metric) pair: **left** -- (A), (B), and (C) histograms overlaid, dotted vertical lines at each distribution's mean; **right** -- how little (A) and (B) differ from the true reference (C), quantified as ECDF(A)-ECDF(C) and ECDF(B)-ECDF(C), with each curve's Kolmogorov-Smirnov statistic/p-value/percent mean difference against (C) annotated in the title.

In [ ]:
# --- Pool the saved per-repeat metrics across the 3 offsets, per (dye, photon level) ---
METRIC_KEYS = [f'{q}_{m}' for q in ('sigma', 'N', 'fpdist') for m in MODELS]


def pooled_metrics(dye, n_photon):
    conds = [load_condition(dye, n_photon, off, keys=METRIC_KEYS) for off in offsets_frac]
    return {k: np.concatenate([c[k] for c in conds]) for k in METRIC_KEYS}


metrics = {dye: pooled_metrics(dye, mid_photon) for dye in dyes}

print(f"pooled {len(offsets_frac) * N_REPEATS} draws per model per dye (n_photon={mid_photon:.0f})")
for dye in dyes:
    m = metrics[dye]
    nan_pct = {k: 100 * np.isnan(m[f'sigma_{k}']).mean() for k in MODELS}
    fp_nan_pct = {k: 100 * np.isnan(m[f'fpdist_{k}']).mean() for k in MODELS}
    print(f"  {dye}: Gaussian-fit NaN rate A/B/C = {nan_pct['A']:.1f}%/{nan_pct['B']:.1f}%/{nan_pct['C']:.1f}%, "
          f"fingerprint NaN rate A/B/C = {fp_nan_pct['A']:.1f}%/{fp_nan_pct['B']:.1f}%/{fp_nan_pct['C']:.1f}%")

In [ ]:
# --- Figure: histogram (A/B/C overlaid) | ECDF-difference vs (C) (+ KS statistics) ---
# One figure per dye (3 rows x 2 cols) rather than one 9-row figure -- keeps each well within
# the publication-plotting framework's max figure depth (8.25"), and each dye's comparison is
# independently readable.
metric_specs = [
    ('sigma_PSF', 'sigma_A', 'sigma_B', 'sigma_C', 'sigma_PSF / nm'),
    ('N_photoelectrons', 'N_A', 'N_B', 'N_C', 'N_pe'),
    ('fingerprint distance', 'fpdist_A', 'fpdist_B', 'fpdist_C', 'fingerprint distance from truth'),
]

ks_rows = []
for dye in dyes:
    m = metrics[dye]
    fig, axs = plotter.two_column_plot(nrows=len(metric_specs), ncols=2, width=6.69, height=2.2 * len(metric_specs))

    for row, (name, key_A, key_B, key_C, xlabel) in enumerate(metric_specs):
        a, b, c = m[key_A], m[key_B], m[key_C]
        a, b, c = a[~np.isnan(a)], b[~np.isnan(b)], c[~np.isnan(c)]  # drop rare fit-rejection NaNs
        ax_hist, ax_diff = axs[row, 0], axs[row, 1]
        label = xlabel

        lo, hi = min(a.min(), b.min(), c.min()), max(a.max(), b.max(), c.max())
        bins = np.linspace(lo, hi, 40)

        ax_hist.hist(a, bins=bins, alpha=0.45, color='tab:blue', density=True, label='(A) factorised')
        ax_hist.hist(b, bins=bins, alpha=0.35, color='tab:orange', density=True, label='(B) photon-by-photon')
        ax_hist.hist(c, bins=bins, density=True, histtype='step', lw=1.5, color='0.2', label='(C) diffraction')
        ax_hist.axvline(a.mean(), color='tab:blue', ls=':', lw=1.3)
        ax_hist.axvline(b.mean(), color='tab:orange', ls=':', lw=1.3)
        ax_hist.axvline(c.mean(), color='0.2', ls=':', lw=1.3)
        ax_hist.set_xlabel(label, fontsize=7)
        ax_hist.set_ylabel('density', fontsize=7)
        ax_hist.tick_params(labelsize=6)
        ax_hist.set_title(name, fontsize=7)
        if row == 0:
            ax_hist.legend(fontsize=6, loc='upper right')

        ks_AC, ks_p_AC = ks_2samp(a, c)
        ks_BC, ks_p_BC = ks_2samp(b, c)
        grid = np.linspace(lo, hi, 400)
        ecdf_a = np.searchsorted(np.sort(a), grid, side='right') / len(a)
        ecdf_b = np.searchsorted(np.sort(b), grid, side='right') / len(b)
        ecdf_c = np.searchsorted(np.sort(c), grid, side='right') / len(c)
        ax_diff.plot(grid, (ecdf_a - ecdf_c) * 100, color='tab:blue', lw=1, label='A - C')
        ax_diff.plot(grid, (ecdf_b - ecdf_c) * 100, color='tab:orange', lw=1, label='B - C')
        ax_diff.axhline(0, color='0.6', lw=0.6)
        pct_diff_AC = 100 * (a.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
        pct_diff_BC = 100 * (b.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
        ax_diff.set_title(f"A-C: KS={ks_AC:.3f} (p={ks_p_AC:.2f}), {pct_diff_AC:+.2f}%   "
                           f"B-C: KS={ks_BC:.3f} (p={ks_p_BC:.2f}), {pct_diff_BC:+.2f}%", fontsize=6)
        ax_diff.set_xlabel(label, fontsize=7)
        ax_diff.set_ylabel('ECDF diff vs (C) / %', fontsize=7)
        ax_diff.tick_params(labelsize=6)
        if row == 0:
            ax_diff.legend(fontsize=6, loc='upper right')

        ks_rows.append({'dye': dye, 'metric': name,
                         'ks_AvsC': ks_AC, 'ks_p_AvsC': ks_p_AC, 'ks_BvsC': ks_BC, 'ks_p_BvsC': ks_p_BC,
                         'mean_A': a.mean(), 'mean_B': b.mean(), 'mean_C': c.mean(),
                         'pct_diff_AvsC': pct_diff_AC, 'pct_diff_BvsC': pct_diff_BC})

    fig.suptitle(dye, fontsize=8, y=1.02)
    plt.savefig(FIGURE_DIR / f"distributional_comparison_{dye.replace(' ', '-')}.svg", bbox_inches='tight')
    plt.show()

ks_df = pd.DataFrame(ks_rows)
ks_df.to_csv(OUT_ROOT / "ks_comparison.csv", index=False)
ks_df

## Photon-count dependence: is the factorised model's error vs. the true optics worse at low or high photon counts?

The distributional comparison above only ran at the single middle photon level -- informative
about distribution *shape*, but a single level can't say whether that's close to the worst
case or not. Repeats the same per-repeat metrics (`sigma_PSF`, `N_photoelectrons`,
fingerprint distance) across *every* level in `photon_levels`, pooling the 3 offsets at
each level, to characterise (A) factorised's and (B) photon-by-photon's error against (C) the
true diffraction reference as a function of photon budget, rather than assuming it from one
point.

In [ ]:
# --- Same pooled per-repeat metrics as above, across ALL photon levels (read from disk) ---
metric_specs_all = [
    ('sigma_PSF', 'sigma_A', 'sigma_B', 'sigma_C'),
    ('N_photoelectrons', 'N_A', 'N_B', 'N_C'),
    ('fingerprint distance', 'fpdist_A', 'fpdist_B', 'fpdist_C'),
]

level_rows = []
for dye in dyes:
    for n_photon in photon_levels:
        m = pooled_metrics(dye, n_photon)
        for name, key_A, key_B, key_C in metric_specs_all:
            a, b, c = m[key_A], m[key_B], m[key_C]
            a, b, c = a[~np.isnan(a)], b[~np.isnan(b)], c[~np.isnan(c)]  # drop rare fit-rejection NaNs
            ks_AB, ks_p_AB = ks_2samp(a, b)
            ks_AC, ks_p_AC = ks_2samp(a, c)
            ks_BC, ks_p_BC = ks_2samp(b, c)
            pct_diff_AB = 100 * (a.mean() - b.mean()) / b.mean() if b.mean() != 0 else float('nan')
            pct_diff_AC = 100 * (a.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
            pct_diff_BC = 100 * (b.mean() - c.mean()) / c.mean() if c.mean() != 0 else float('nan')
            level_rows.append({
                'dye': dye, 'n_photon': n_photon, 'metric': name,
                'mean_A': a.mean(), 'mean_B': b.mean(), 'mean_C': c.mean(),
                'pct_diff_AvsB': pct_diff_AB, 'pct_diff_AvsC': pct_diff_AC, 'pct_diff_BvsC': pct_diff_BC,
                'ks_AvsB': ks_AB, 'ks_p_AvsB': ks_p_AB,
                'ks_AvsC': ks_AC, 'ks_p_AvsC': ks_p_AC, 'ks_BvsC': ks_BC, 'ks_p_BvsC': ks_p_BC,
                'n_fit_A': len(a), 'n_fit_B': len(b), 'n_fit_C': len(c),
            })

level_df = pd.DataFrame(level_rows)
level_df.to_csv(OUT_ROOT / "pct_mean_diff_vs_photon_count.csv", index=False)
worst_AB = level_df.loc[level_df['pct_diff_AvsB'].abs().idxmax()]
worst_AC = level_df.loc[level_df['pct_diff_AvsC'].abs().idxmax()]
worst_BC = level_df.loc[level_df['pct_diff_BvsC'].abs().idxmax()]
print(f"worst |A vs B| (factorised vs photon-by-photon, same Gaussian form): {worst_AB['pct_diff_AvsB']:.3f}% "
      f"at n_photon={worst_AB['n_photon']:.0f}, metric={worst_AB['metric']}, dye={worst_AB['dye']}")
print(f"worst |A vs C| (factorised vs true diffraction): {worst_AC['pct_diff_AvsC']:.3f}% "
      f"at n_photon={worst_AC['n_photon']:.0f}, metric={worst_AC['metric']}, dye={worst_AC['dye']}")
print(f"worst |B vs C| (photon-by-photon Gaussian vs true diffraction): {worst_BC['pct_diff_BvsC']:.3f}% "
      f"at n_photon={worst_BC['n_photon']:.0f}, metric={worst_BC['metric']}, dye={worst_BC['dye']}")
sig_rows = level_df[level_df.metric == 'sigma_PSF']
worst_n_fit_frac = (sig_rows[['n_fit_A', 'n_fit_B', 'n_fit_C']].min(axis=1) / (len(offsets_frac) * N_REPEATS)).min()
print(f"worst per-condition Gaussian-fit success rate (sigma_PSF rows): {worst_n_fit_frac*100:.1f}%")
level_df.pivot_table(index=['metric', 'n_photon'], columns='dye', values='pct_diff_AvsC').round(3)

In [ ]:
# --- Headline figure: does the production factorised model (A) stay close to the true
# diffraction-based optics reference (C)? Top row: absolute metric value vs photon count,
# (A) solid markers, (C) dotted -- do the lines sit on top of each other? Bottom row: percent
# deviation of (A) from (C), with a shaded +/-5% band as a visual "not too bad" reference.
fig, axs = plotter.two_column_plot(nrows=2, ncols=3, width=6.69, height=4.4)

dye_colours = {'ATTO 488': 'tab:blue', 'ATTO 565': 'tab:green', 'ATTO 647N': 'tab:red'}
metric_labels = {
    'sigma_PSF': r'$\sigma_{\rm PSF}$ / nm',
    'N_photoelectrons': r'$N_{\rm pe}$',
    'fingerprint distance': 'fingerprint distance\nfrom truth',
}

for col, (name, _, _, _) in enumerate(metric_specs_all):
    sub = level_df[level_df.metric == name]
    ax_val, ax_pct = axs[0, col], axs[1, col]

    for dye in dyes:
        d = sub[sub.dye == dye].sort_values('n_photon')
        ax_val.plot(d['n_photon'], d['mean_C'], lw=2.2, ls=':', color=dye_colours[dye], alpha=0.6,
                    label=f'{dye} — (C) true diffraction' if col == 0 else None)
        ax_val.plot(d['n_photon'], d['mean_A'], marker='o', ms=3, lw=1.2, ls='-', color=dye_colours[dye],
                    label=f'{dye} — (A) factorised (production)' if col == 0 else None)
        ax_pct.plot(d['n_photon'], d['pct_diff_AvsC'], marker='o', ms=3, lw=1.2, color=dye_colours[dye])

    ax_val.set_xscale('log')
    ax_val.set_xlabel('photons / punctum', fontsize=8)
    ax_val.set_ylabel(metric_labels[name], fontsize=8)
    ax_val.set_title(name, fontsize=8)
    if name in ('N_photoelectrons', 'fingerprint distance'):
        ax_val.set_yscale('log')  # falls roughly as 1/sqrt(N)
    ax_val.tick_params(labelsize=7)

    ax_pct.axhspan(-5, 5, color='0.85', zorder=0)
    ax_pct.axhline(0, color='0.5', lw=0.7)
    ax_pct.set_xscale('log')
    ax_pct.set_xlabel('photons / punctum', fontsize=8)
    ax_pct.set_ylabel('(A - C) / C  %', fontsize=8)
    ax_pct.tick_params(labelsize=7)

axs[0, 0].legend(fontsize=5, loc='best')
fig.suptitle('Production factorised model (A) vs. the true diffraction-based reference (C)', fontsize=8, y=1.02)
plt.savefig(FIGURE_DIR / "model_comparison_vs_photon_count.svg", bbox_inches='tight')
plt.show()

In [ ]:
# --- Headline figure: does the production factorised model (A) stay close to the
# photon-by-photon reference (B)? Both are Gaussian -- this isolates the effect of
# wavelength-resolving alone, with no diffraction-vs-Gaussian question involved. Top row:
# absolute metric value vs photon count, (A) solid markers, (B) dotted -- do the lines sit on
# top of each other? Bottom row: percent deviation of (A) from (B), with a shaded +/-5% band
# as a visual "not too bad" reference.
fig, axs = plotter.two_column_plot(nrows=2, ncols=3, height=5)

dye_colours = {'ATTO 488': 'tab:blue', 'ATTO 565': 'tab:green', 'ATTO 647N': 'tab:red'}
metric_labels = {
    'sigma_PSF': r'$\sigma_{\rm PSF}$ / nm',
    'N_photoelectrons': r'$N_{\rm pe}$',
    'fingerprint distance': 'fingerprint distance\nfrom truth',
}

for col, (name, _, _, _) in enumerate(metric_specs_all):
    sub = level_df[level_df.metric == name]
    ax_val, ax_pct = axs[0, col], axs[1, col]

    for dye in dyes:
        d = sub[sub.dye == dye].sort_values('n_photon')
        ax_val.plot(d['n_photon'], d['mean_B'], lw=2.2, ls=':', color=dye_colours[dye], alpha=0.6,
                    label=f'{dye} — (B) photon-by-photon' if col == 0 else None)
        ax_val.plot(d['n_photon'], d['mean_A'], marker='o', ms=3, lw=1.2, ls='-', color=dye_colours[dye],
                    label=f'{dye} — (A) factorised (production)' if col == 0 else None)
        ax_pct.plot(d['n_photon'], d['pct_diff_AvsB'], marker='s', ms=3, lw=1.2, color=dye_colours[dye])

    ax_val.set_xscale('log')
    ax_val.set_xlabel('photons / punctum', fontsize=8)
    ax_val.set_ylabel(metric_labels[name], fontsize=8)
    if name in ('N_photoelectrons', 'fingerprint distance'):
        ax_val.set_yscale('log')  # N spans two decades; the distance falls roughly as 1/sqrt(N)
    ax_val.tick_params(labelsize=7)
    ax_val.grid(True, lw=0.25, alpha=0.5, ls='--', color='gray')

    ax_pct.axhspan(-5, 5, color='0.85', alpha=0.5, zorder=0)
    ax_pct.axhline(0, color='0.5', lw=0.7)
    ax_pct.set_xscale('log')
    ax_pct.set_xlabel('photons / punctum', fontsize=8)
    ax_pct.set_ylabel('(A - B) / B  %', fontsize=8)
    ax_pct.tick_params(labelsize=7)
    ax_pct.grid(True, lw=0.25, alpha=0.5, ls='--', color='gray')

axs[0, 0].legend(fontsize=5, loc='best')
plt.savefig(FIGURE_DIR / "model_comparison_vs_photon_count_AvsB.svg", bbox_inches='tight')
plt.show()

## Wavelength-bin convergence check (model C)

Model (C) discretises each dye's emission spectrum into `N_WAVELENGTH_BINS` representative
wavelengths for spatial photon placement (see "Model (C)" above) -- a computational
compromise, not a physical assumption. Before trusting the `sigma_PSF` gap reported above,
check whether that gap is sensitive to bin count: rerun (C) with several times more bins, for
the single (dye, photon level) condition with the largest measured |A vs C| gap (the condition
where any binning artifact would be most visible, since the fit is best resolved there), and
compare the resulting `sigma_PSF` mean against the original. If the shift is small relative to
the reported gap, bin count is not a meaningful driver of the headline result.

In [ ]:
# --- Wavelength-bin convergence check: does increasing N_WAVELENGTH_BINS shift model (C)'s
# sigma_PSF? Targeted, not a full rerun -- the single (dye, n_photon) condition with the
# largest measured |A vs C| gap, pooled across the 3 offsets. Temporarily overwrites
# diffraction_cache for that one dye/offset combination and restores it afterward.
N_WAVELENGTH_BINS_CHECK = 4 * N_WAVELENGTH_BINS

sig_only = level_df[level_df.metric == 'sigma_PSF']
check_dye = sig_only.loc[sig_only['pct_diff_AvsC'].abs().idxmax(), 'dye']
check_photon = photon_levels[-1]
rng_check = np.random.default_rng(RNG_SEED + 999)

sigma_C_orig_list, sigma_C_check_list = [], []
for (fx, fy) in offsets_frac:
    sigma_C_orig_list.append(load_condition(check_dye, check_photon, (fx, fy), keys=['sigma_C'])['sigma_C'])

    dye_spectrum = S_F.get_dye_or_filter_data(names=[check_dye], wavelength=wavelength, dye_or_filter=True)[0]
    bin_centres_wl, bin_edges_wl = build_wavelength_bins(dye_spectrum, wavelength, N_WAVELENGTH_BINS_CHECK)
    patches_check = vpsf.compute_psf_stack(
        z_offsets_um=np.array([0.0]),
        wavelengths_um=bin_centres_wl / 1000.0,
        x0_um=fx * (PIXEL_SIZE_NM / 1000.0),
        y0_um=fy * (PIXEL_SIZE_NM / 1000.0),
    )[0]

    old_entry = diffraction_cache[(check_dye, (fx, fy))]
    diffraction_cache[(check_dye, (fx, fy))] = (patches_check, bin_edges_wl)
    try:
        img_C_check = simulate_diffraction(check_dye, check_photon, N_REPEATS, camera_parameters_base, rng_check, (fx, fy))
    finally:
        diffraction_cache[(check_dye, (fx, fy))] = old_entry

    sigma_C_check_list.append(fit_sigma_nm(fit_stack(camera_image(img_C_check, camera_parameters_base))))

sigma_C_orig = np.concatenate(sigma_C_orig_list)
sigma_C_check = np.concatenate(sigma_C_check_list)
sigma_C_orig = sigma_C_orig[~np.isnan(sigma_C_orig)]
sigma_C_check = sigma_C_check[~np.isnan(sigma_C_check)]

pct_shift = 100 * (sigma_C_check.mean() - sigma_C_orig.mean()) / sigma_C_orig.mean()
orig_gap = sig_only.loc[(sig_only.dye == check_dye) & (sig_only.n_photon == check_photon), 'pct_diff_AvsC'].iloc[0]
print(f"Wavelength-bin convergence check -- {check_dye}, n_photon={check_photon:.0f} "
      f"(largest measured |A vs C| sigma_PSF gap: {orig_gap:+.2f}%)")
print(f"  sigma_C mean, N_WAVELENGTH_BINS={N_WAVELENGTH_BINS} (original): {sigma_C_orig.mean():.2f} nm")
print(f"  sigma_C mean, N_WAVELENGTH_BINS={N_WAVELENGTH_BINS_CHECK} (finer):    {sigma_C_check.mean():.2f} nm")
print(f"  shift from finer bins: {pct_shift:+.3f}% "
      f"({'small' if abs(pct_shift) < 0.2 * abs(orig_gap) else 'NOT small'} relative to the {orig_gap:+.2f}% A-vs-C gap)")